In [1]:
#1 loading all .txt files from knowledge_base
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter    # ← corrected import

loader = DirectoryLoader("knowledge_base", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
documents = loader.load()
print(f"Loaded {len(documents)} documents")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(documents)
print(f"Total chunks: {len(docs)}")

C:\Users\MARSHENIL\AppData\Local\Temp\ipykernel_7580\2404313130.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Loaded 4 documents
Total chunks: 12


Loaded 4 documents – my four .txt files in knowledge_base were found and read successfully.

Total chunks: 12 – after splitting with 500-character chunks and 50-character overlap, those four files produced 12 meaningful text chunks. This is perfectly normal for a compact knowledge base. It will still demonstrate retrieval perfectly.

The deprecation warning about langchain-community is irrelevant – it won’t affect anything.

In [2]:
#2 creating vector store with sentence embeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(docs, embedding_model, persist_directory="chroma_db")

print("Vector store saved to chroma_db")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store saved to chroma_db


HF Hub warning – I m  not logged into Hugging Face, so I m using an anonymous (rate‑limited) connection. That’s fine; the model is already downloaded and cached. No effect on project.

Loading weights – the embedding model (all-MiniLM-L6-v2) is being loaded into memory. It found the cached files, so it only took a moment.

Vector store saved to chroma_db – Chroma created and persisted the vector database. My knowledge base is now searchable.

In [3]:
#3 function to retrieve relevant guidelines and generate a diagnostic report
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import ollama

# reload the persisted vector store
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory="chroma_db", embedding_function=embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def generate_report(anomaly_description):
    # retrieve most relevant guideline chunks
    docs_retrieved = retriever.invoke(anomaly_description)
    context = "\n".join([d.page_content for d in docs_retrieved])

    # build the prompt
    prompt = f"""You are a satellite fault diagnostic assistant. Based on the anomaly description and the following fault recovery guidelines, write a concise report with probable cause and recommended actions.

Anomaly: {anomaly_description}

Guidelines:
{context}

Diagnostic Report:"""

    # call the local LLM
    response = ollama.chat(model="llama3:8b", messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

The embedding model loaded successfully, and the generate_report function is now defined and ready to use.

The Loading weights line is from the HuggingFaceEmbeddings model being loaded into memory, which is required for retrieval. 

In [4]:
test_desc = "At 2022-06-02 14:30 UTC, sensor CADC0872 (magnetometer) showed a sudden drop to near-zero for several consecutive readings, while other channels remained normal."
report = generate_report(test_desc)
print(report)

Report ID: 20220602-1430-CADC0872

Probable Cause: Sensor disconnection or power cycling is the likely cause of the sudden drop to near-zero values in sensor CADC0872 (magnetometer) at 14:30 UTC on June 2, 2022.

Recommended Actions:

1. Switch to redundant magnetometer to ensure normal operations.
2. Verify power bus voltage to rule out any potential issues with the power supply.
3. If the signal remains absent after switching to the redundant magnetometer, flag for ground-based recalibration to resolve the issue and restore nominal operation of the magnetometer.

Further investigation or action may be required if the problem persists or is not resolved by these recommended actions.


The anomaly description was sent to the generate_report function.

ChromaDB retrieved the most relevant chunks from your knowledge base (likely the “Magnetometer Anomaly” guideline).

Ollama (Llama 3) used that context to produce a coherent, structured report.

The whole process ran 100% offline on my laptop, proving the local LLM works.